Lloyds Banking Group – Task 2
Customer Churn Prediction Model

1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', None)

2. Load Dataset

In [3]:
file_path ="../Loyds/smart_bank_churn_project/data/Customer_Churn_Data_Large (1).xlsx"

customer_demo = pd.read_excel(file_path, sheet_name='Customer_Demographics')
transaction_history = pd.read_excel(file_path, sheet_name='Transaction_History')
customer_service = pd.read_excel(file_path, sheet_name='Customer_Service')
online_activity = pd.read_excel(file_path, sheet_name='Online_Activity')
churn_status = pd.read_excel(file_path, sheet_name='Churn_Status')

3. Aggregate Transaction Data

In [4]:
transaction_agg = transaction_history.groupby('CustomerID').agg({
    'AmountSpent': ['sum', 'mean', 'count']
}).reset_index()

transaction_agg.columns = [
    'CustomerID',
    'TotalSpent',
    'AverageSpent',
    'TransactionCount'
]

transaction_agg.head()

,CustomerID,TotalSpent,AverageSpent,TransactionCount
0,1,416.50,416.50000,1
1,2,1547.42,221.06000,7
2,3,1702.98,283.83000,6
3,4,917.29,183.45800,5
4,5,2001.49,250.18625,8


4. Aggregate Customer Service Data

In [5]:
service_agg = customer_service.groupby('CustomerID').agg({
    'InteractionID': 'count'
}).reset_index()

service_agg.columns = [
    'CustomerID',
    'ServiceInteractionCount'
]

service_agg.head()

,CustomerID,ServiceInteractionCount
0,1,1
1,2,1
2,3,1
3,4,2
4,6,1


5. Merge All Datasets

In [6]:
merged_data = customer_demo.merge(
    transaction_agg,
    on='CustomerID',
    how='left'
)

merged_data = merged_data.merge(
    service_agg,
    on='CustomerID',
    how='left'
)

merged_data = merged_data.merge(
    online_activity,
    on='CustomerID',
    how='left'
)

merged_data = merged_data.merge(
    churn_status,
    on='CustomerID',
    how='left'
)

merged_data.head()

,CustomerID,Age,Gender,MaritalStatus,IncomeLevel,TotalSpent,AverageSpent,TransactionCount,ServiceInteractionCount,LastLoginDate,LoginFrequency,ServiceUsage,ChurnStatus
0,1,62,M,Single,Low,416.50,416.50000,1,1.0,2023-10-21,34,Mobile App,0
1,2,65,M,Married,Low,1547.42,221.06000,7,1.0,2023-12-05,5,Website,1
2,3,18,M,Single,Low,1702.98,283.83000,6,1.0,2023-11-15,3,Website,0
3,4,21,M,Widowed,Low,917.29,183.45800,5,2.0,2023-08-25,2,Website,0
4,5,21,M,Divorced,Medium,2001.49,250.18625,8,NaN,2023-10-27,41,Website,0


In [ ]:
6. Handle Missing Values

In [7]:
merged_data['ServiceInteractionCount'] = (
    merged_data['ServiceInteractionCount'].fillna(0)
)

7. Encode Categorical Variables

In [8]:
categorical_columns = [
    'Gender',
    'MaritalStatus',
    'IncomeLevel',
    'ServiceUsage'
]

merged_data_encoded = pd.get_dummies(
    merged_data,
    columns=categorical_columns,
    drop_first=True
)

merged_data_encoded.head()

,CustomerID,Age,TotalSpent,AverageSpent,TransactionCount,ServiceInteractionCount,LastLoginDate,LoginFrequency,ChurnStatus,Gender_M,MaritalStatus_Married,MaritalStatus_Single,MaritalStatus_Widowed,IncomeLevel_Low,IncomeLevel_Medium,ServiceUsage_Online Banking,ServiceUsage_Website
0,1,62,416.50,416.50000,1,1.0,2023-10-21,34,0,True,False,True,False,True,False,False,False
1,2,65,1547.42,221.06000,7,1.0,2023-12-05,5,1,True,True,False,False,True,False,False,True
2,3,18,1702.98,283.83000,6,1.0,2023-11-15,3,0,True,False,True,False,True,False,False,True
3,4,21,917.29,183.45800,5,2.0,2023-08-25,2,0,True,False,False,True,True,False,False,True
4,5,21,2001.49,250.18625,8,0.0,2023-10-27,41,0,True,False,False,False,False,True,False,True


8. Drop Unnecessary Columns

In [9]:
merged_data_encoded = merged_data_encoded.drop(
    columns=['CustomerID', 'LastLoginDate']
)

merged_data_encoded.head()

,Age,TotalSpent,AverageSpent,TransactionCount,ServiceInteractionCount,LoginFrequency,ChurnStatus,Gender_M,MaritalStatus_Married,MaritalStatus_Single,MaritalStatus_Widowed,IncomeLevel_Low,IncomeLevel_Medium,ServiceUsage_Online Banking,ServiceUsage_Website
0,62,416.50,416.50000,1,1.0,34,0,True,False,True,False,True,False,False,False
1,65,1547.42,221.06000,7,1.0,5,1,True,True,False,False,True,False,False,True
2,18,1702.98,283.83000,6,1.0,3,0,True,False,True,False,True,False,False,True
3,21,917.29,183.45800,5,2.0,2,0,True,False,False,True,True,False,False,True
4,21,2001.49,250.18625,8,0.0,41,0,True,False,False,False,False,True,False,True


9. Define Features and Target

In [10]:
X = merged_data_encoded.drop('ChurnStatus', axis=1)

y = merged_data_encoded['ChurnStatus']

10. Train Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

11. Feature Scaling

In [12]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

12. Logistic Regression Model

In [13]:
log_model = LogisticRegression()

log_model.fit(X_train, y_train)

LogisticRegression()

13. Predictions

In [14]:
y_pred = log_model.predict(X_test)

y_prob = log_model.predict_proba(X_test)[:,1]

14. Model Evaluation

In [15]:
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC AUC Score:", roc_auc)

Accuracy: 0.795
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
ROC AUC Score: 0.4844301273201411


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
